this notebook is for simulating live streaming using scraped messeges

In [1]:
#setup

import json
from kafka import KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError
import time
import json
import sqlite3
from kafka import KafkaProducer

with open("config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

BOOTSTRAP_SERVER = config["kafka"]["bootstrap_server"]
RAW_TOPIC = config["kafka"]["raw_topic"]
TOKENS_TOPIC = config["kafka"]["tokens_topic"]
RAW_PARTITIONS = config["kafka"].get("raw_partitions", 1)
TOKENS_PARTITIONS = config["kafka"].get("tokens_partitions", 4)

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVER)
topics = [
    NewTopic(name=RAW_TOPIC, num_partitions=RAW_PARTITIONS, replication_factor=1),
    NewTopic(name=TOKENS_TOPIC, num_partitions=TOKENS_PARTITIONS, replication_factor=1)
]
#check topics / create
for t in topics:
    try:
        admin.create_topics([t], validate_only=False)
        print(f"Topic '{t.name}' created.")
    except TopicAlreadyExistsError:
        print(f"Topic '{t.name}' already exists.")
admin.close()

Topic 'telegram-raw' already exists.
Topic 'telegram-tokens' already exists.


In [2]:
producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVER,
    value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode('utf-8')
)

DB_PATH = config["messages_db_path"]

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
#here we can replace what messages we want to send, for example from a certine day.
#this can help us evaluate the system
#if we know about an event at a certine date and time, we can simulate that day to see if  we caught the trend live or not
cursor.execute("SELECT channel, sender, text, ts FROM messages ORDER BY ts ASC LIMIT 10000")
rows = cursor.fetchall()
conn.close()

print(f"Loaded {len(rows)} messages from {DB_PATH}.")
print(f"Streaming at accelerated speed into topic '{RAW_TOPIC}'...")

#Add delay in messeges if needed. the system works without them 
PUSH_DELAY = 0.005 

try:
    for channel, sender, text, ts in rows:
        payload = {
            "channel": channel,
            "sender": sender,
            "text": text,
            "ts": ts
        }
        producer.send(RAW_TOPIC, value=payload)
        time.sleep(PUSH_DELAY)
        
    producer.flush()
    print("All historical messages replayed successfully")
except KeyboardInterrupt:
    print("\nReplay stopped by user")
finally:
    producer.close()

Loaded 8699 messages from messages.db.
Streaming at accelerated speed into topic 'telegram-raw'...
All historical messages replayed successfully
